# 📊 Notebook 01: Exploratory Data Analysis (EDA)

**Goal**: Understand the dataset before any modeling.

Checklist:
- [ ] Dataset shape, dtypes, missing values
- [ ] Time series visualization (trend, seasonality, noise)
- [ ] Stationarity test (ADF, KPSS)
- [ ] Autocorrelation plots (ACF, PACF)
- [ ] Distribution analysis
- [ ] Outlier detection

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')  # if running from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

from src.data.loader import DataLoader

plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 50)
print('✅ Libraries loaded')

In [ ]:
# ── 1. Load Data ──────────────────────────────────────────────────────
# TODO: Update path to your Kaggle dataset
loader = DataLoader()

# Option A: Local file
df = loader.load_csv('data/raw/train.csv')  # UPDATE THIS

# Option B: Download from Kaggle (requires ~/.kaggle/kaggle.json)
# loader.download_kaggle(competition='<competition-name>')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── 2. Basic Info ──────────────────────────────────────────────────────
# UPDATE: Set your actual column names
DATE_COL = 'date'    # ← UPDATE
TARGET_COL = 'value' # ← UPDATE

print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
# ── 3. Time Series Plot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df[DATE_COL], df[TARGET_COL], linewidth=1.0, color='#2196F3')
ax.set_title(f'Time Series: {TARGET_COL}', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel(TARGET_COL)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('results/plots/01_timeseries_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4. Seasonal Decomposition ──────────────────────────────────────────
# UPDATE: period = 7 (weekly), 12 (monthly), 365 (yearly)
period = 7  # ← UPDATE based on your data frequency

decomp = seasonal_decompose(df.set_index(DATE_COL)[TARGET_COL].dropna(),
                            model='additive', period=period)
fig = decomp.plot()
fig.set_size_inches(16, 10)
plt.suptitle('Seasonal Decomposition', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/plots/01_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5. ACF / PACF ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df[TARGET_COL].dropna(), ax=axes[0], lags=50, title='ACF')
plot_pacf(df[TARGET_COL].dropna(), ax=axes[1], lags=50, title='PACF')
plt.tight_layout()
plt.savefig('results/plots/01_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Stationarity Tests ───────────────────────────────────────────────
series = df[TARGET_COL].dropna()

# ADF Test
adf_result = adfuller(series)
print('=== ADF Test ===')
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print(f'Stationary?   : {"Yes ✅" if adf_result[1] < 0.05 else "No ❌ → apply differencing"}')

# KPSS Test
try:
    kpss_result = kpss(series, regression='c', nlags='auto')
    print('\n=== KPSS Test ===')
    print(f'KPSS Statistic: {kpss_result[0]:.4f}')
    print(f'p-value       : {kpss_result[1]:.4f}')
    print(f'Stationary?   : {"Yes ✅" if kpss_result[1] > 0.05 else "No ❌"}')
except Exception as e:
    print(f'KPSS error: {e}')

In [ ]:
# ── 7. Distribution ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET_COL].dropna(), bins=50, color='#2196F3', edgecolor='white')
axes[0].set_title('Distribution', fontsize=12)
import scipy.stats as stats
stats.probplot(df[TARGET_COL].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot', fontsize=12)
plt.tight_layout()
plt.savefig('results/plots/01_distribution.png', dpi=150, bbox_inches='tight')
plt.show()